# Defect Formation Energy of a Boron Vacancy in h-BN

> **Fabian Bertoldo, Sajid Ali, Simone Manti & Kristian S. Thygesen**,
> "Quantum point defects in 2D materials - the QPOD database", npj Computational Materials, 2022.
> [DOI:10.1038/s41524-022-00730-w](https://doi.org/10.1038/s41524-022-00730-w)

Computes the neutral formation energy of the vacancy created in the
[structure notebook](defect_point_vacancy_boron_nitride.ipynb) and compares it with QPOD's
`v_B in BN (charge 0)` entry, 10.18 eV at standard states, using the generic
[Defect Formation Energy](../workflows/defect_formation_energy.ipynb) workflow. The SCF is fixed to
the doublet state QPOD reports.

`RELAX_DEFECT = False` (default) runs the SCF only; `RELAX_DEFECT = True` relaxes the defective
cell first, for the paper's result.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")


### 1.2. Material names

In [ ]:
# Names saved by defect_point_vacancy_boron_nitride.ipynb.
PRISTINE_NAME = "h-BN supercell"
DEFECTIVE_NAME = "B-vacancy h-BN"


### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
DEFECT_WORKFLOW_SEARCH_TERM = "defect_formation_energy.json"
MY_WORKFLOW_NAME = "Defect Formation Energy"
APPLICATION_NAME = "espresso"

# False: SCF only, ~7 min. True: relax the defective cell first (~1 h) -- needed to reproduce
# the paper's value.
RELAX_DEFECT = False

CLUSTER_NAME = "cluster-001"
QUEUE_NAME = QueueName.OF
PPN = 40
TIME_LIMIT = "12:00:00"  # covers the optional relaxation (~1 h measured)

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds


### 1.4. DFT model parameters

In [ ]:
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the platform's default PBE family for B and N
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

# QPOD: 6 Å⁻¹ for relaxations, 12 for ground states; 6 used here for cost -- the neutral E_f
# does not need the denser grid.
KPOINT_DENSITY = 6
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry"

SCF_UNIT = "pw_scf"
RELAX_UNIT = "pw_relax"
SPIN_SETTINGS = {"nspin": 2, "tot_magnetization": 1}  # doublet QPOD reports for the neutral V_B
RELAXATION_SETTINGS = {"forc_conv_thr": 3.9e-4, "nstep": 100}  # 0.01 eV/Å, QPOD's convergence target
ELEMENTAL_ENERGY_SOURCE = "my_account"


## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account

In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load the materials
### 3.1. Load from the uploads folder, and print provenance

In [ ]:
from collections import Counter
from mat3ra.notebooks_utils.material import load_material_from_folder

def formula(material):
    counts = Counter(material.basis.elements.values)
    return "".join(f"{element}{counts[element]}" for element in sorted(counts))

materials_by_name = {}
for name in (PRISTINE_NAME, DEFECTIVE_NAME):
    material = load_material_from_folder(FOLDER, name, verbose=False)
    if material is None:
        raise ValueError(f"No material named '{name}' in '{FOLDER}'. Run "
                         "defect_point_vacancy_boron_nitride.ipynb first, or correct the name above.")
    materials_by_name[name] = material

pristine, defective = materials_by_name[PRISTINE_NAME], materials_by_name[DEFECTIVE_NAME]
for name, material in materials_by_name.items():
    a, b = material.lattice.a, material.lattice.b
    print(f"{name}: {formula(material)}, {material.basis.number_of_atoms} atoms, "
          f"cell {a:.2f} x {b:.2f} Å, min in-plane vector {min(a, b):.2f} Å")


### 3.2. Resolve elemental reference materials

Elemental references are platform materials, not Standata entries: the workflow resolves
`{'tags': 'elemental', ...}` and `total_energy` by the material's `exabyteId`, never shared by an upload.

In [ ]:
from mat3ra.made.material import Material

elements = sorted(set(pristine.basis.elements.values) | set(defective.basis.elements.values))

elemental_materials_data = client.materials.list({"tags": "elemental", "metadata.element": {"$in": elements}})
elemental_materials = {}
for element in elements:
    matches = [m for m in elemental_materials_data if m.get("metadata", {}).get("element") == element]
    if not matches:
        raise ValueError(f"No platform elemental reference material tagged for {element}; "
                         "seed one (tags=['elemental'], metadata.element) first.")
    elemental_materials[element] = matches[0]
    atoms = Material.create(matches[0]).basis.number_of_atoms
    print(f"{element}: {matches[0]['name']} ({matches[0]['_id']}, owner {matches[0]['owner']['slug']}, "
          f"{atoms} atoms)")


### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_pristine = get_or_create_material(client, pristine, ACCOUNT_ID)
saved_defective = get_or_create_material(client, defective, ACCOUNT_ID)


## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
print(f"Using application: {app.name}, model: {MODEL_TAG}")


### 4.2. k-grid per material

In [ ]:
import math

# made.Material objects, purely for lattice/basis access -- kept alongside the platform dicts
# in elemental_materials, which are used for ids and job creation.
material_objects = {PRISTINE_NAME: pristine, DEFECTIVE_NAME: defective,
                    **{element: Material.create(data) for element, data in elemental_materials.items()}}

def kgrid_for_density(material, periodic_dims=(0, 1, 2)):
    # |b_i| = 2*pi*reciprocal_vector_norms[i]; dims outside periodic_dims stay at 1 (vacuum).
    norms = material.lattice.reciprocal_vector_norms
    grid = [1, 1, 1]
    for dim in periodic_dims:
        grid[dim] = max(1, math.ceil(KPOINT_DENSITY * 2 * math.pi * norms[dim]))
    return grid

kgrid = {
    PRISTINE_NAME: kgrid_for_density(pristine, periodic_dims=(0, 1)),
    DEFECTIVE_NAME: kgrid_for_density(defective, periodic_dims=(0, 1)),
    **{element: kgrid_for_density(material_objects[element]) for element in elemental_materials},
}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")


## 5. Configure compute
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")


## 6. Prerequisite Total Energy jobs

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_scf_kgrid
from mat3ra.notebooks_utils.job import create_job


def run_or_reuse(workflow, materials, label, kind="Total Energy"):
    # Keyed on this exact name (model-aware); jobs on curators-owned materials (elementals) are allowed.
    existing = client.jobs.list(
        {"_material._id": materials[0]["_id"], "owner._id": ACCOUNT_ID, "status": "finished",
         "workflow.name": workflow.name},
        {"sort": {"updatedAt": -1}, "limit": 1},
    )
    if existing:
        print(f"♻️  {label}: reusing existing {kind} job {existing[0]['_id']}")
        return existing[0]["_id"], False
    job_response = create_job(
        api_client=client, materials=materials, workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, prefix=f"{workflow.name} {timestamp}", compute=compute.to_dict(),
    )
    print(f"✅ {label}: created {kind} job {job_response['_id']}")
    return job_response["_id"], True


# The pristine's own reference is resolved separately, by highest precision among this
# material's qe: total_energy properties -- the results cell's E_f (recomputed) line is the check.
total_energy_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)
prerequisite_materials = {PRISTINE_NAME: saved_pristine, **elemental_materials}
prerequisite_job_ids = {}
new_job_ids = []
for name, saved_material in prerequisite_materials.items():
    workflow = Workflow.create(total_energy_workflow_config)
    workflow.name = f"Total Energy {name} {MODEL_TAG}"
    subworkflow = workflow.subworkflows[0]
    subworkflow.model = model
    unit = subworkflow.get_unit_by_name(name=SCF_UNIT)
    unit.add_context(cutoffs_context)
    subworkflow.set_unit(unit)
    apply_scf_kgrid(workflow, kgrid[name], material=material_objects[name])
    job_id, created = run_or_reuse(workflow, [saved_material], name)
    prerequisite_job_ids[name] = job_id
    if created:
        new_job_ids.append(job_id)


In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} prerequisite job(s).")
    await wait_for_jobs_to_finish_async(client.jobs, new_job_ids, poll_interval=POLL_INTERVAL)


## 7. Relax the defective cell (optional)

Runs only if `RELAX_DEFECT`, and is reused on a rerun like the prerequisites.

In [ ]:
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job

relaxed_defective = None
if RELAX_DEFECT:
    relax_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
        RELAX_WORKFLOW_SEARCH_TERM
    )
    relax_workflow = Workflow.create(relax_workflow_config)
    relax_workflow.name = f"Fixed-cell Relaxation {DEFECTIVE_NAME} {MODEL_TAG}"
    relax_subworkflow = relax_workflow.subworkflows[0]
    relax_subworkflow.model = model
    relax_unit = relax_subworkflow.get_unit_by_name(name=RELAX_UNIT)
    relax_unit.add_context(cutoffs_context)
    relax_subworkflow.set_unit(relax_unit)
    apply_scf_kgrid(relax_workflow, kgrid[DEFECTIVE_NAME], material=defective, unit_name=RELAX_UNIT)
    patch_workflow_qe_input(relax_workflow, {"system": SPIN_SETTINGS}, [RELAX_UNIT])
    patch_workflow_qe_input(relax_workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])
    relax_job_id, created = run_or_reuse(
        relax_workflow, [saved_defective], DEFECTIVE_NAME, kind="Fixed-cell Relaxation"
    )
    if created:
        submit_jobs(client.jobs, [relax_job_id])
        await wait_for_jobs_to_finish_async(client.jobs, [relax_job_id], poll_interval=POLL_INTERVAL)

    props = get_properties_for_job(client, relax_job_id, "final_structure")
    relaxed_defective = Material.create(client.materials.get(props[-1]["materialId"]))
    print(f"Relaxed defective material: {relaxed_defective.name} ({relaxed_defective.id}), "
          f"{formula(relaxed_defective)}, {relaxed_defective.basis.number_of_atoms} atoms")

    total_force = get_properties_for_job(client, relax_job_id, "total_force")[0]
    print(f"Residual force after relaxation (norm over all atoms): "
          f"{total_force['value']:.4f} {total_force['units']}")


## 8. Configure the Defect Formation Energy workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

defect_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    DEFECT_WORKFLOW_SEARCH_TERM
)
defect_workflow = Workflow.create(defect_workflow_config)
defect_workflow.name = f"{MY_WORKFLOW_NAME} {MODEL_TAG}" + (" relaxed" if RELAX_DEFECT else "")

for subworkflow in defect_workflow.subworkflows:
    if subworkflow.name == "Compute Total Energy for Defective Material":
        subworkflow.model = model
        unit = subworkflow.get_unit_by_name(name=SCF_UNIT)
        unit.add_context(cutoffs_context)
        subworkflow.set_unit(unit)
    elif subworkflow.name == "Resolve Total Energies for Elemental Materials":
        source_unit = subworkflow.get_unit_by_name(name="assign-source-of-te-for-an-element")
        source_unit.value = f"'{ELEMENTAL_ENERGY_SOURCE}'"
        subworkflow.set_unit(source_unit)
apply_scf_kgrid(defect_workflow, kgrid[DEFECTIVE_NAME], material=defective)
patch_workflow_qe_input(defect_workflow, {"system": SPIN_SETTINGS}, unit_names=[SCF_UNIT])

visualize_workflow(defect_workflow)


## 9. Create and run the Defect Formation Energy job
### 9.1. Create the job (defective + pristine)

In [ ]:
defect_materials = [relaxed_defective, saved_pristine] if RELAX_DEFECT else [saved_defective, saved_pristine]
defect_job_response = create_job(
    api_client=client, materials=defect_materials, workflow=defect_workflow,
    project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
    prefix=f"{MY_WORKFLOW_NAME} {formula(defective)} {timestamp}",
)
defect_job_id = defect_job_response["_id"]
print(f"✅ Defect Formation Energy job created: {defect_job_id}")


### 9.2. Submit and monitor the job

In [ ]:
client.jobs.submit(defect_job_id)
print(f"✅ Job {defect_job_id} submitted successfully!")
await wait_for_jobs_to_finish_async(client.jobs, [defect_job_id], poll_interval=POLL_INTERVAL)


## 10. Retrieve the results
### 10.1. Defect formation energy

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

defect_energy_data = get_properties_for_job(client, defect_job_id, property_name="defect_formation_energy")
visualize_properties(defect_energy_data, title="Defect Formation Energy")
e_formation = defect_energy_data[0]["value"]

def total_energy_for(job_id):
    return get_properties_for_job(client, job_id, property_name="total_energy")[0]["value"]

# Consistency check: recompute E_f from the three total energies this notebook owns.
e_def = total_energy_for(defect_job_id)
e_pris = total_energy_for(prerequisite_job_ids[PRISTINE_NAME])
b_atoms = material_objects["B"].basis.number_of_atoms
e_b_per_atom = total_energy_for(prerequisite_job_ids["B"]) / b_atoms
e_formation_check = e_def - e_pris + e_b_per_atom
print(f"μ_B = {e_b_per_atom:.4f} eV/atom ({b_atoms} atoms)")
print(f"E_f (workflow): {e_formation:.3f} eV, E_f (recomputed): {e_formation_check:.3f} eV")


### 10.2. Compare with QPOD

Only the neutral (q = 0) defect is compared.

In [ ]:
QPOD = {"standard_states": 10.18, "B_poor": 8.89}  # eV, QPOD v_B in BN (charge 0)

difference = e_formation - QPOD["standard_states"]
verdict = "yes" if abs(difference) <= 0.2 else "no"  # 0.2 eV: pseudopotential set and cell size
config_label = "relaxed defect" if RELAX_DEFECT else "unrelaxed SCF"
print(f"E_f (this notebook):        {e_formation:.3f} eV")
print(f"E_f (QPOD, standard states): {QPOD['standard_states']:.3f} eV")
print(f"E_f (QPOD, B-poor):          {QPOD['B_poor']:.3f} eV")
print(f"Difference from standard states: {difference:+.3f} eV")
print(f"Reproduces Bertoldo et al. (2022): {verdict} ({config_label})")


## References

[1] Bertoldo, Ali, Manti & Thygesen, "Quantum point defects in 2D materials — the QPOD database", npj Comput. Mater. 8, 56 (2022). https://doi.org/10.1038/s41524-022-00730-w
[2] QPOD entry v_B in BN (charge 0): https://qpod.fysik.dtu.dk/material/1BN-1.2d.v_B.0.1